In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

# Dizionario di parole iniziali troppo generiche per fare merge al Passo 1
STOP_FIRST_WORDS = {
    "UNIV", "INST", "NATIONAL", "CHINA", "BEIJING", "SHANGHAI", "JAPAN", "NIPPON", "KOREA",
    "GENERAL", "AMERICAN", "INTERNATIONAL", "UNITED", "ROYAL", "STATE", "NEW", "NORTH", "SOUTH",
    "EAST", "WEST", "ADVANCED", "APPLIED", "GLOBAL", "FIRST", "TECH", "QUANTUM", "DIGITAL",
    "SMART", "BIO", "MED", "PHARMA", "ENERGY", "SOLAR", "POWER", "MICRO", "NANO", "OPTO",
    "ELECTRO", "AUTO", "AERO", "CHEM", "POLITECNICO", "ECOLE", "HOCHSCHULE", "COLLEGE",
    "SCHOOL", "ACADEMY", "CENTER", "LABORATORY", "LAB", "FOUNDATION", "MINISTRY", "AGENCY",
    "COUNCIL", "HOSPITAL", "ARMY", "NAVY", "AIR", "GE", "AT"
}

def pipeline_armonizzazione(series_clean, stop_words=STOP_FIRST_WORDS, min_len=3, alpha_word=0.6, distance_threshold=0.72):
    s_clean = series_clean.fillna("").astype(str).str.strip()
    first_word = s_clean.str.split().str[0].fillna("")
    
    # Conta quante righe hanno quella stessa prima parola
    fw_counts = first_word.value_counts()
    
    # --- PARTE 1: Identifica chi ha prima parola uguale e valida ---
    # Condizioni: lunghezza >= min_len, non numero, non in stop_words, presente almeno 2 volte
    is_valid_fw = (
        (first_word.str.len() >= min_len) &
        (~first_word.str.isdigit()) &
        (~first_word.isin(stop_words)) &
        (first_word.map(fw_counts) >= 2)
    )
    
    # --- PARTE 2: Match diretto -> tiene solo la prima parola come nome unico ---
    final_name = pd.Series(index=s_clean.index, dtype="object")
    final_name[is_valid_fw] = first_word[is_valid_fw]
    
    # --- PARTE 3: TF-IDF + Clustering sulle righe NON mergiate ---
    unmerged = s_clean[~is_valid_fw]
    unique_unmerged = unmerged.unique()
    
    if len(unique_unmerged) > 1:
        # Similarità ibrida (parole + n-grammi di caratteri per catturare refusi)
        vec_w = TfidfVectorizer(analyzer="word", token_pattern=r"(?u)\b\w+\b")
        sim_w = cosine_similarity(vec_w.fit_transform(unique_unmerged))
        
        vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4))
        sim_c = cosine_similarity(vec_c.fit_transform(unique_unmerged))
        
        dist = 1 - (alpha_word * sim_w + (1 - alpha_word) * sim_c)
        np.fill_diagonal(dist, 0)
        
        clust = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=distance_threshold
        )
        c_ids = clust.fit_predict(dist)
        
        # Mappa ogni nome unico al suo cluster ID
        u_to_cid = dict(zip(unique_unmerged, c_ids))
        unmerged_cids = unmerged.map(u_to_cid)
        
        # Sceglie il nome finale del cluster TF-IDF (il più frequente; a parità, il più corto)
        df_u = pd.DataFrame({"clean": unmerged, "cid": unmerged_cids})
        def best_label(s):
            vc = s.value_counts()
            top = vc[vc == vc.iloc[0]].index.tolist()
            return min(top, key=len)
        
        cid_to_name = df_u.groupby("cid")["clean"].agg(best_label)
        final_name[~is_valid_fw] = unmerged_cids.map(cid_to_name)
        
    elif len(unique_unmerged) == 1:
        final_name[~is_valid_fw] = unique_unmerged[0]
        
    return final_name, is_valid_fw

# --- ESEMPIO DI UTILIZZO ---
# df["clean"] = df["comp"].map(clean)  # tua funzione clean già pronta
# df["nome_finale"], df["mergiato_step1"] = pipeline_armonizzazione(df["clean"])